# 03 - Models (Logistic Regression, Decision Tree, KNN)

In [1]:
import sys
from pathlib import Path

REPO_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))

import numpy as np
import pandas as pd
import joblib

from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import StratifiedKFold, GridSearchCV
from sklearn.metrics import (
    precision_score, recall_score, f1_score, average_precision_score,
    confusion_matrix, classification_report,
)

from src.preprocessing import NUMERIC_COLS, CATEGORICAL_COLS
from src.features import numeric_cols_for_scenario_c

pd.set_option("display.max_columns", 30)
RANDOM_STATE = 42

## Load feature splits

In [2]:
splits = joblib.load(REPO_ROOT / "results" / "feature_splits.joblib")

y_train = splits["y_train"]
y_test = splits["y_test"]
modes_train = splits["modes_train"]
modes_test = splits["modes_test"]

scenarios = {
    "A": (splits["X_train_A"], splits["X_test_A"], NUMERIC_COLS),
    "B": (splits["X_train_B"], splits["X_test_B"],
          [c for c in splits["X_train_B"].columns if c not in CATEGORICAL_COLS]),
    "C": (splits["X_train_C"], splits["X_test_C"], numeric_cols_for_scenario_c(add_osf_margin=True)),
}

for name, (xtr, xte, num_cols) in scenarios.items():
    print(name, xtr.shape, xte.shape, len(num_cols))


A (8000, 6) (2000, 6) 5
B (8000, 21) (2000, 21) 20
C (8000, 10) (2000, 10) 9


## Build model pipelines

In [3]:
def make_column_transformer(numeric_cols, scale_numeric):
    numeric_step = StandardScaler() if scale_numeric else "passthrough"
    return ColumnTransformer(
        transformers=[
            ("num", numeric_step, numeric_cols),
            ("cat", OneHotEncoder(handle_unknown="ignore"), CATEGORICAL_COLS),
        ]
    )

MODEL_SPECS = {
    "logreg": {
        "estimator": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
        "scale": True,
        "param_grid": {"clf__C": [0.01, 0.1, 1.0, 10.0]},
    },
    "tree": {
        "estimator": DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
        "scale": False,
        "param_grid": {"clf__max_depth": [3, 5, 8, None], "clf__min_samples_leaf": [1, 5, 10, 20]},
    },
    "knn": {
        "estimator": KNeighborsClassifier(),
        "scale": True,
        "param_grid": {"clf__n_neighbors": [3, 5, 7, 11, 15], "clf__weights": ["uniform", "distance"]},
    },
}

def build_pipeline(model_key, numeric_cols):
    spec = MODEL_SPECS[model_key]
    preprocessor = make_column_transformer(numeric_cols, spec["scale"])
    return Pipeline([("prep", preprocessor), ("clf", spec["estimator"])])


## Tune and evaluate every model on every scenario

In [4]:
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

results_rows = []
fitted_pipelines = {}

for scenario_name, (X_train, X_test, numeric_cols) in scenarios.items():
    for model_key in MODEL_SPECS:
        pipeline = build_pipeline(model_key, numeric_cols)
        grid = GridSearchCV(
            pipeline,
            param_grid=MODEL_SPECS[model_key]["param_grid"],
            scoring="average_precision",
            cv=cv,
            n_jobs=-1,
        )
        grid.fit(X_train, y_train)
        best_model = grid.best_estimator_

        y_pred = best_model.predict(X_test)
        y_score = best_model.predict_proba(X_test)[:, 1]

        row = {
            "scenario": scenario_name,
            "model": model_key,
            "best_params": grid.best_params_,
            "precision": precision_score(y_test, y_pred, zero_division=0),
            "recall": recall_score(y_test, y_pred, zero_division=0),
            "f1": f1_score(y_test, y_pred, zero_division=0),
            "pr_auc": average_precision_score(y_test, y_score),
        }
        results_rows.append(row)
        fitted_pipelines[(scenario_name, model_key)] = best_model

results_df = pd.DataFrame(results_rows)
results_df


,scenario,model,best_params,precision,recall,f1,pr_auc
0,A,logreg,{'clf__C': 0.1},0.138889,0.808824,0.237069,0.379759
1,A,tree,"{'clf__max_depth': 8, 'clf__min_samples_leaf': 5}",0.419118,0.838235,0.558824,0.731281
2,A,knn,"{'clf__n_neighbors': 15, 'clf__weights': 'dist...",0.909091,0.147059,0.253165,0.559680
3,B,logreg,{'clf__C': 10.0},0.218750,0.926471,0.353933,0.505687
4,B,tree,"{'clf__max_depth': 8, 'clf__min_samples_leaf':...",0.347059,0.867647,0.495798,0.742229
5,B,knn,"{'clf__n_neighbors': 15, 'clf__weights': 'dist...",0.944444,0.250000,0.395349,0.642257
6,C,logreg,{'clf__C': 0.1},0.169492,0.882353,0.284360,0.440469
7,C,tree,"{'clf__max_depth': 5, 'clf__min_samples_leaf':...",0.391304,0.926471,0.550218,0.876297
8,C,knn,"{'clf__n_neighbors': 15, 'clf__weights': 'dist...",0.952381,0.294118,0.449438,0.657626


## Baseline: always predict no failure

In [5]:
baseline_pred = np.zeros_like(y_test)
baseline_row = {
    "scenario": "-",
    "model": "majority_class",
    "best_params": {},
    "precision": precision_score(y_test, baseline_pred, zero_division=0),
    "recall": recall_score(y_test, baseline_pred, zero_division=0),
    "f1": f1_score(y_test, baseline_pred, zero_division=0),
    "pr_auc": y_test.mean(),
}
pd.DataFrame([baseline_row])


,scenario,model,best_params,precision,recall,f1,pr_auc
0,-,majority_class,{},0.0,0.0,0.0,0.034


## Confusion matrices, best model per scenario

In [6]:
for scenario_name in scenarios:
    sub = results_df[results_df["scenario"] == scenario_name].sort_values("f1", ascending=False)
    best_model_key = sub.iloc[0]["model"]
    model = fitted_pipelines[(scenario_name, best_model_key)]
    X_test = scenarios[scenario_name][1]
    y_pred = model.predict(X_test)

    print(scenario_name, best_model_key)
    print(confusion_matrix(y_test, y_pred))
    print(classification_report(y_test, y_pred, zero_division=0))


A tree
[[1853   79]
 [  11   57]]
              precision    recall  f1-score   support

           0       0.99      0.96      0.98      1932
           1       0.42      0.84      0.56        68

    accuracy                           0.95      2000
   macro avg       0.71      0.90      0.77      2000
weighted avg       0.97      0.95      0.96      2000

B tree
[[1821  111]
 [   9   59]]
              precision    recall  f1-score   support

           0       1.00      0.94      0.97      1932
           1       0.35      0.87      0.50        68

    accuracy                           0.94      2000
   macro avg       0.67      0.91      0.73      2000
weighted avg       0.97      0.94      0.95      2000

C tree
[[1834   98]
 [   5   63]]
              precision    recall  f1-score   support

           0       1.00      0.95      0.97      1932
           1       0.39      0.93      0.55        68

    accuracy                           0.95      2000
   macro avg       0.69   

## Recall per failure mode, test set

In [7]:
def recall_per_mode(y_pred, modes_test):
    rows = []
    for mode in modes_test.columns:
        mask = modes_test[mode] == 1
        if mask.sum() == 0:
            rows.append({"mode": mode, "n": 0, "recall": np.nan})
            continue
        caught = y_pred[mask.values] == 1
        rows.append({"mode": mode, "n": int(mask.sum()), "recall": float(caught.mean())})
    return pd.DataFrame(rows)

mode_rows = []
for scenario_name in scenarios:
    for model_key in MODEL_SPECS:
        model = fitted_pipelines[(scenario_name, model_key)]
        X_test = scenarios[scenario_name][1]
        y_pred = model.predict(X_test)
        df_mode = recall_per_mode(y_pred, modes_test)
        df_mode["scenario"] = scenario_name
        df_mode["model"] = model_key
        mode_rows.append(df_mode)

mode_recall_df = pd.concat(mode_rows, ignore_index=True)
mode_recall_df.pivot_table(index=["scenario", "model"], columns="mode", values="recall")


mode                  HDF     OSF       PWF   RNF  TWF
scenario model                                        
A        knn     0.034483  0.2500  0.384615  0.00  0.0
         logreg  0.827586  1.0000  1.000000  0.25  0.3
         tree    0.965517  1.0000  0.923077  0.50  0.3
B        knn     0.068966  0.5625  0.461538  0.00  0.0
         logreg  0.931034  1.0000  0.923077  0.50  0.9
         tree    0.931034  1.0000  0.923077  0.25  0.6
C        knn     0.172414  0.5625  0.538462  0.00  0.0
         logreg  0.965517  1.0000  0.923077  0.25  0.5
         tree    1.000000  1.0000  1.000000  0.25  0.7

## Save models and results

In [8]:
MODELS_DIR = REPO_ROOT / "models"
MODELS_DIR.mkdir(exist_ok=True)
RESULTS_DIR = REPO_ROOT / "results"
RESULTS_DIR.mkdir(exist_ok=True)

for (scenario_name, model_key), model in fitted_pipelines.items():
    joblib.dump(model, MODELS_DIR / f"{model_key}_{scenario_name}.joblib")

results_df.to_csv(RESULTS_DIR / "metrics_scenarios.csv", index=False)
mode_recall_df.to_csv(RESULTS_DIR / "recall_per_mode.csv", index=False)

sorted(p.name for p in MODELS_DIR.glob("*.joblib"))


['knn_A.joblib',
 'knn_B.joblib',
 'knn_C.joblib',
 'logreg_A.joblib',
 'logreg_B.joblib',
 'logreg_C.joblib',
 'tree_A.joblib',
 'tree_B.joblib',
 'tree_C.joblib']